In [ ]:
import os
import cv2
import numpy as np
import librosa
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import librosa
from tensorflow.keras.preprocessing.sequence import pad_sequences

def extract_frames(video_path, num_frames=30):
    try:
        cap = cv2.VideoCapture(video_path)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_interval = max(total_frames // num_frames, 1)

        for i in range(num_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * frame_interval)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (224, 224))
                frames.append(frame)
            else:
                break

        cap.release()
        if len(frames) == 0:
            raise ValueError("No frames extracted")
        return np.array(frames)
    except Exception as e:
        print(f"Error extracting frames from {video_path}: {e}")
        return None

def extract_audio_features(audio_path, n_mfcc=40, max_pad_len=862):
    try:
        y, sr = librosa.load(audio_path, sr=None)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        if mfcc.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :max_pad_len]
        return mfcc.T
    except Exception as e:
        print(f"Error extracting audio features from {audio_path}: {e}")
        return None

# Paths to dataset
DATASET_PATH = '/content/drive/MyDrive/Emotion Dataset/Video/split_video/test'
CLASS_NAMES = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

video_data = []
audio_data = []
labels = []

for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_PATH, class_name)
    for video_name in os.listdir(class_path):
        video_path = os.path.join(class_path, video_name)
        if not video_name.endswith('.mp4'):
            continue
        frames = extract_frames(video_path)
        audio_features = extract_audio_features(video_path)

        if frames is not None and audio_features is not None:
            video_data.append(frames)
            audio_data.append(audio_features)
            labels.append(CLASS_NAMES.index(class_name))
        else:
            print(f"Skipping {video_name} due to insufficient data")

# Pad video sequences to a uniform length
video_data_padded = pad_sequences(video_data, padding='post', dtype='float32')

audio_data = pad_sequences(audio_data, padding='post', dtype='float32')  # Pad sequences to the same length
labels = np.array(labels)

print("Video data shape:", video_data_padded.shape) # Use the padded video data
print("Audio data shape:", audio_data.shape)
print("Labels shape:", labels.shape)


<ipython-input-10-bcf2f83b9898>:20: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None)
/usr/local/lib/python3.10/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
<ipython-input-10-bcf2f83b9898>:20: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None)
/usr/local/lib/python3.10/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
<ipython-input-10-bcf2f83b9898>:20: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None)
/usr/local/lib/python3.10/dist-packages/libro

Video data shape: (74, 30, 224, 224, 3)
Audio data shape: (74, 862, 40)
Labels shape: (74,)


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
import tensorflow as tf

# Load pre-trained MobileNetV2 model + higher level layers
base_model = MobileNetV2(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
video_model = Model(inputs=base_model.input, outputs=x)

# Extract features from each frame set
video_features = []
for frames in video_data:
    features = video_model.predict(frames)
    video_features.append(features.mean(axis=0))  # Average features over frames

video_features = np.array(video_features)
print("Video features shape:", video_features.shape)


1/1 [==============================] - 0s 21ms/step
Video features shape: (74, 1280)


In [ ]:
from sklearn.model_selection import train_test_split

X_train_video, X_test_video, X_train_audio, X_test_audio, y_train, y_test = train_test_split(
    video_features, audio_data, labels, test_size=0.3, random_state=42)

# Ensure audio data shape compatibility
X_train_audio = np.expand_dims(X_train_audio, -1)
X_test_audio = np.expand_dims(X_test_audio, -1)

print("Train video shape:", X_train_video.shape)
print("Test video shape:", X_test_video.shape)
print("Train audio shape:", X_train_audio.shape)
print("Test audio shape:", X_test_audio.shape)
print("Train labels shape:", y_train.shape)
print("Test labels shape:", y_test.shape)

Train video shape: (51, 1280)
Test video shape: (23, 1280)
Train audio shape: (51, 862, 40, 1)
Test audio shape: (23, 862, 40, 1)
Train labels shape: (51,)
Test labels shape: (23,)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, concatenate

# Video model
video_input = Input(shape=(1280,))  # Assuming 1280 feature dimensions from MobileNetV2

# Audio model
audio_input = Input(shape=(862, 40))
audio_x = LSTM(128)(audio_input)

# Fusion model
merged = concatenate([video_input, audio_x])
dense_1 = Dense(128, activation='relu')(merged)
output = Dense(len(CLASS_NAMES), activation='softmax')(dense_1)

fusion_model = Model(inputs=[video_input, audio_input], outputs=output)
fusion_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
fusion_model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 862, 40)]            0         []                            
                                                                                                  
 input_2 (InputLayer)        [(None, 1280)]               0         []                            
                                                                                                  
 lstm (LSTM)                 (None, 128)                  86528     ['input_3[0][0]']             
                                                                                                  
 concatenate (Concatenate)   (None, 1408)                 0         ['input_2[0][0]',             
                                                                     'lstm[0][0]']          

In [ ]:
# Train the model
history = fusion_model.fit(
    [X_train_video, X_train_audio], y_train,
    validation_data=([X_test_video, X_test_audio], y_test),
    epochs=10, batch_size=2)

Epoch 1/10
26/26 [==============================] - 4s 57ms/step - loss: 2.6504 - accuracy: 0.1176 - val_loss: 3.2994 - val_accuracy: 0.0435
Epoch 2/10
26/26 [==============================] - 1s 39ms/step - loss: 2.1594 - accuracy: 0.1569 - val_loss: 2.2672 - val_accuracy: 0.1304
Epoch 3/10
26/26 [==============================] - 1s 47ms/step - loss: 1.8662 - accuracy: 0.2549 - val_loss: 2.3185 - val_accuracy: 0.0870
Epoch 4/10
26/26 [==============================] - 1s 42ms/step - loss: 1.6600 - accuracy: 0.2745 - val_loss: 2.4750 - val_accuracy: 0.0435
Epoch 5/10
26/26 [==============================] - 1s 42ms/step - loss: 1.4877 - accuracy: 0.4118 - val_loss: 2.4665 - val_accuracy: 0.0870
Epoch 6/10
26/26 [==============================] - 1s 32ms/step - loss: 1.3888 - accuracy: 0.4902 - val_loss: 2.5050 - val_accuracy: 0.0870
Epoch 7/10
26/26 [==============================] - 1s 32ms/step - loss: 1.4215 - accuracy: 0.3922 - val_loss: 2.6418 - val_accuracy: 0.0870
Epoch 8/10
26

In [ ]:
fusion_model.save('/content/drive/MyDrive/Emotion Dataset/bangla_emotion_detection_model.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
# Generate classification report
from sklearn.metrics import classification_report
y_pred = np.argmax(fusion_model.predict([X_test_video, X_test_audio]), axis=1)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

1/1 [==============================] - 1s 639ms/step
              precision    recall  f1-score   support

       Angry       0.17      0.20      0.18         5
     Disgust       0.00      0.00      0.00         3
        Fear       0.00      0.00      0.00         2
       Happy       1.00      0.20      0.33         5
     Neutral       0.00      0.00      0.00         1
         Sad       0.40      0.50      0.44         4
    Surprise       0.00      0.00      0.00         3

    accuracy                           0.17        23
   macro avg       0.22      0.13      0.14        23
weighted avg       0.32      0.17      0.19        23



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
from tensorflow.keras.models import load_model

# Load the trained model
model_path = '/content/drive/MyDrive/Emotion Dataset/bangla_emotion_detection_model.h5'
fusion_model = load_model(model_path)


In [ ]:
import numpy as np

def preprocess_video(video_path, num_frames=30, n_mfcc=40, max_pad_len=862):
    frames = extract_frames(video_path, num_frames)
    audio_features = extract_audio_features(video_path, n_mfcc, max_pad_len)

    if frames.shape[0] > 0 and audio_features.shape[0] > 0:
        # Extract video features
        video_features = video_model.predict(frames)
        video_features = video_features.mean(axis=0)  # Average features over frames
        video_features = np.expand_dims(video_features, axis=0)  # Add batch dimension

        # Expand audio features to add batch dimension
        audio_features = np.expand_dims(audio_features, axis=0)
        return video_features, audio_features
    else:
        raise ValueError("Insufficient data in the video or audio")

# Test preprocessing
test_video_path = '/content/drive/MyDrive/emotion small test/Happy/P_10_H_1(1).mp4'
video_features, audio_features = preprocess_video(test_video_path)


<ipython-input-10-bcf2f83b9898>:20: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None)
/usr/local/lib/python3.10/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


1/1 [==============================] - 0s 85ms/step


In [ ]:
# Predict the emotion
predicted_probabilities = fusion_model.predict([video_features, audio_features])
predicted_class = np.argmax(predicted_probabilities, axis=1)
predicted_emotion = CLASS_NAMES[predicted_class[0]]

print(f"The predicted emotion for the input video is: {predicted_emotion}")

1/1 [==============================] - 1s 562ms/step
The predicted emotion for the input video is: Happy


In [ ]:
# prompt: tensorflow virsion

import tensorflow as tf

print(tf.__version__)


2.15.0


In [ ]:
# prompt: tensorflow install specific virsion in pip

!pip install tensorflow==2.9
